In [1]:
import pandas as pd
import numpy as np
import joblib
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, recall_score
from imblearn.over_sampling import SMOTE

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
model = joblib.load('fraud_model.pkl')

week1 = pd.read_csv('week1_baseline.csv')
week2 = pd.read_csv('week2_drift.csv')
week3 = pd.read_csv('week3_drift.csv')
week4 = pd.read_csv('week4_drift.csv')

print("Model and datasets loaded successfully!")

Model and datasets loaded successfully!


In [3]:
# Define thresholds
KS_THRESHOLD = 0.1
PSI_THRESHOLD = 0.2
AUC_THRESHOLD = 0.95

print("Thresholds set:")
print(f"KS Test threshold:  {KS_THRESHOLD}")
print(f"PSI threshold:      {PSI_THRESHOLD}")
print(f"AUC-ROC threshold:  {AUC_THRESHOLD}")
print("\nIf any threshold is crossed - retraining will trigger automatically!")

Thresholds set:
KS Test threshold:  0.1
PSI threshold:      0.2
AUC-ROC threshold:  0.95

If any threshold is crossed - retraining will trigger automatically!


In [4]:
def check_and_retrain(new_data, week_name):
    print(f"\n{'='*50}")
    print(f"Checking {week_name}...")
    print(f"{'='*50}")
    
    # Check KS Test
    ks_stat, _ = stats.ks_2samp(week1['V1'], new_data['V1'])
    print(f"KS Statistic: {ks_stat:.4f} (threshold: {KS_THRESHOLD})")
    
    # Check PSI
    base_counts, bin_edges = np.histogram(week1['V1'], bins=10)
    current_counts, _ = np.histogram(new_data['V1'], bins=bin_edges)
    base_pct = base_counts / len(week1) + 1e-6
    current_pct = current_counts / len(new_data) + 1e-6
    psi = np.sum((current_pct - base_pct) * np.log(current_pct / base_pct))
    print(f"PSI Score:    {psi:.4f} (threshold: {PSI_THRESHOLD})")
    
    # Check AUC
    X = new_data.drop('Class', axis=1)
    y = new_data['Class']
    y_pred = model.predict(X)
    auc = roc_auc_score(y, y_pred)
    print(f"AUC-ROC:      {auc:.4f} (threshold: {AUC_THRESHOLD})")
    
    # Trigger retraining if any threshold crossed
    if ks_stat > KS_THRESHOLD or psi > PSI_THRESHOLD or auc < AUC_THRESHOLD:
        print(f"\n🚨 DRIFT DETECTED - Retraining triggered for {week_name}!")
        
        # Retrain model
        X_train = new_data.drop('Class', axis=1)
        y_train = new_data['Class']
        
        smote = SMOTE(random_state=42)
        X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
        
        new_model = RandomForestClassifier(n_estimators=10, random_state=42)
        new_model.fit(X_train_sm, y_train_sm)
        
        new_auc = roc_auc_score(y, new_model.predict(X))
        print(f"✅ Retraining complete!")
        print(f"Old AUC: {auc:.4f} → New AUC: {new_auc:.4f}")
        
        return new_model
    else:
        print(f"✅ No drift detected - No retraining needed!")
        return model

print("Retraining function created successfully!")

Retraining function created successfully!


In [5]:
print("Running Adaptive Drift Monitor...")
print("Checking all weeks for drift and retraining if needed...")

model_week2 = check_and_retrain(week2, "Week 2 - Slight Drift")
model_week3 = check_and_retrain(week3, "Week 3 - Moderate Drift")
model_week4 = check_and_retrain(week4, "Week 4 - Severe Drift")


Running Adaptive Drift Monitor...
Checking all weeks for drift and retraining if needed...

Checking Week 2 - Slight Drift...
KS Statistic: 0.2776 (threshold: 0.1)
PSI Score:    0.0187 (threshold: 0.2)
AUC-ROC:      0.9674 (threshold: 0.95)

🚨 DRIFT DETECTED - Retraining triggered for Week 2 - Slight Drift!
✅ Retraining complete!
Old AUC: 0.9674 → New AUC: 1.0000

Checking Week 3 - Moderate Drift...
KS Statistic: 0.5765 (threshold: 0.1)
PSI Score:    0.3518 (threshold: 0.2)
AUC-ROC:      0.9883 (threshold: 0.95)

🚨 DRIFT DETECTED - Retraining triggered for Week 3 - Moderate Drift!
✅ Retraining complete!
Old AUC: 0.9883 → New AUC: 1.0000

Checking Week 4 - Severe Drift...
KS Statistic: 0.7457 (threshold: 0.1)
PSI Score:    0.8409 (threshold: 0.2)
AUC-ROC:      0.9255 (threshold: 0.95)

🚨 DRIFT DETECTED - Retraining triggered for Week 4 - Severe Drift!
✅ Retraining complete!
Old AUC: 0.9255 → New AUC: 1.0000
